In [ ]:
# Example 1: Basic Schema Validation with CoreSchema
from pydantic_core import SchemaValidator, ValidationError
from pydantic_core import core_schema

print("=== Basic Schema Validation ===")

# Define a simple schema for user data
user_schema = core_schema.typed_dict_schema({
    'name': core_schema.typed_dict_field(core_schema.str_schema()),
    'age': core_schema.typed_dict_field(core_schema.int_schema(ge=0, le=120)),
    'email': core_schema.typed_dict_field(core_schema.str_schema()),
    'is_active': core_schema.typed_dict_field(
        core_schema.with_default_schema(
            core_schema.bool_schema(),
            default=True
        ),
        required=False
    )
})

print(f"Created schema: {type(user_schema)}")

# Create a validator from the schema
validator = SchemaValidator(user_schema)
print(f"Created validator: {type(validator)}")

# Test valid data
valid_data = {
    'name': 'Alice Johnson',
    'age': 30,
    'email': 'alice@example.com',
    'is_active': True
}

# Valid data must validate: a ValidationError here fails the cell
validated_data = validator.validate_python(valid_data)
print(f"✓ Valid data validated successfully:")
print(f"  Name: {validated_data['name']}")
print(f"  Age: {validated_data['age']}")
print(f"  Email: {validated_data['email']}")
print(f"  Active: {validated_data['is_active']}")
assert validated_data == valid_data, validated_data

# Test data with missing optional field (should use default)
partial_data = {
    'name': 'Bob Smith',
    'age': 25,
    'email': 'bob@example.com'
}

validated_partial = validator.validate_python(partial_data)
print(f"✓ Partial data with defaults:")
print(f"  Active (default): {validated_partial['is_active']}")
# The omitted optional field comes back with its default
assert validated_partial == {**partial_data, 'is_active': True}, validated_partial

print("Example 1 completed successfully!")

In [ ]:
# Example 2: Complex Schema Validation and Error Handling
import json

print("=== Complex Schema Validation and Error Handling ===")

# Define a more complex nested schema
product_schema = core_schema.typed_dict_schema({
    'id': core_schema.typed_dict_field(core_schema.int_schema(ge=1)),
    'name': core_schema.typed_dict_field(core_schema.str_schema(min_length=1, max_length=100)),
    'price': core_schema.typed_dict_field(core_schema.float_schema(ge=0.0)),
    'tags': core_schema.typed_dict_field(core_schema.list_schema(core_schema.str_schema())),
    'metadata': core_schema.typed_dict_field(core_schema.dict_schema(
        keys_schema=core_schema.str_schema(),
        values_schema=core_schema.any_schema()
    )),
    'category': core_schema.typed_dict_field(core_schema.literal_schema(['electronics', 'books', 'clothing'])),
    'available': core_schema.typed_dict_field(core_schema.bool_schema())
})

validator = SchemaValidator(product_schema)

# Test with valid complex data
valid_product = {
    'id': 101,
    'name': 'Laptop Computer',
    'price': 999.99,
    'tags': ['electronics', 'computers', 'portable'],
    'metadata': {
        'weight': '2.5kg',
        'warranty': 2,
        'specs': {'ram': '16GB', 'storage': '512GB SSD'}
    },
    'category': 'electronics',
    'available': True
}

validated_product = validator.validate_python(valid_product)
print("✓ Complex product validation successful:")
print(f"  Product: {validated_product['name']} (${validated_product['price']})")
print(f"  Tags: {len(validated_product['tags'])} tags")
print(f"  Metadata keys: {list(validated_product['metadata'].keys())}")
assert validated_product == valid_product, validated_product

# Test with invalid data to demonstrate error handling
invalid_products = [
    {
        'id': -1,  # Invalid: negative ID
        'name': 'Test Product',
        'price': 10.0,
        'tags': ['test'],
        'metadata': {},
        'category': 'electronics',
        'available': True
    },
    {
        'id': 102,
        'name': '',  # Invalid: empty name
        'price': 50.0,
        'tags': ['test'],
        'metadata': {},
        'category': 'books',
        'available': True
    },
    {
        'id': 103,
        'name': 'Another Product',
        'price': 25.0,
        'tags': ['test'],
        'metadata': {},
        'category': 'invalid_category',  # Invalid: not in literal options
        'available': True
    }
]

print(f"\nTesting {len(invalid_products)} invalid products:")
expected_first_errors = [('greater_than_equal', ('id',)), ('string_too_short', ('name',)), ('literal_error', ('category',))]
for i, (invalid_product, expected) in enumerate(zip(invalid_products, expected_first_errors, strict=True), 1):
    try:
        validator.validate_python(invalid_product)
    except ValidationError as e:
        error_count = len(e.errors())
        first_error = e.errors()[0]
        print(f"✓ Product {i}: Correctly failed validation ({error_count} errors)")
        print(f"  Rejected first at: {first_error['type']} at {first_error['loc']}")
        assert (first_error['type'], first_error['loc']) == expected, (i, first_error)
    else:
        raise AssertionError(f"Product {i}: invalid product passed validation: {invalid_product}")

print("Example 2 completed successfully!")

In [ ]:
# Example 3: JSON Validation and Type Coercion
from datetime import datetime, timezone
print("=== JSON Validation and Type Coercion ===")

# Schema that demonstrates type coercion capabilities
coercion_schema = core_schema.typed_dict_schema({
    'user_id': core_schema.typed_dict_field(core_schema.int_schema()),  # Should coerce from string if possible
    'score': core_schema.typed_dict_field(core_schema.float_schema()),  # Should coerce from int/string
    'active': core_schema.typed_dict_field(core_schema.bool_schema()),  # Should coerce from various representations
    'created_at': core_schema.typed_dict_field(core_schema.datetime_schema()),  # Should parse datetime strings
    'settings': core_schema.typed_dict_field(core_schema.dict_schema(
        keys_schema=core_schema.str_schema(),
        values_schema=core_schema.union_schema([
            core_schema.int_schema(),
            core_schema.str_schema(),
            core_schema.bool_schema()
        ])
    ))
})

validator = SchemaValidator(coercion_schema)

# Test JSON validation with type coercion
json_data = '''
{
    "user_id": "123",
    "score": "95.5",
    "active": "true",
    "created_at": "2024-01-15T10:30:00",
    "settings": {
        "notifications": "false",
        "max_items": "50",
        "theme": "dark"
    }
}
'''

print("Testing JSON validation with type coercion:")
print(f"Raw JSON length: {len(json_data)} characters")

# Valid JSON must validate: a ValidationError here fails the cell
validated_from_json = validator.validate_json(json_data)
print("✓ JSON validation successful with type coercion:")
print(f"  User ID: {validated_from_json['user_id']} (type: {type(validated_from_json['user_id']).__name__})")
print(f"  Score: {validated_from_json['score']} (type: {type(validated_from_json['score']).__name__})")
print(f"  Active: {validated_from_json['active']} (type: {type(validated_from_json['active']).__name__})")
print(f"  Created: {validated_from_json['created_at']}")
print(f"  Settings: {validated_from_json['settings']}")
assert validated_from_json == {
    'user_id': 123, 'score': 95.5, 'active': True, 'created_at': datetime(2024, 1, 15, 10, 30),
    'settings': {'notifications': 'false', 'max_items': '50', 'theme': 'dark'},
}, validated_from_json
assert [type(validated_from_json[k]) for k in ('user_id', 'score', 'active')] == [int, float, bool]

# Test Python object validation with coercion
python_data = {
    'user_id': '456',  # String that should convert to int
    'score': 87,       # Int that should convert to float
    'active': 1,       # Int that should convert to bool
    'created_at': '2024-02-20T15:45:30Z',  # String that should parse to datetime
    'settings': {
        'notifications': 'true',  # String bool
        'max_items': 100,         # Already int
        'auto_save': False        # Already bool
    }
}

print(f"\nTesting Python object validation with coercion:")
validated_python = validator.validate_python(python_data)
print("✓ Python validation successful with coercion:")
print(f"  User ID coerced: {python_data['user_id']} → {validated_python['user_id']}")
print(f"  Score coerced: {python_data['score']} → {validated_python['score']}")
print(f"  Active coerced: {python_data['active']} → {validated_python['active']}")
assert validated_python == {
    'user_id': 456, 'score': 87.0, 'active': True,
    'created_at': datetime(2024, 2, 20, 15, 45, 30, tzinfo=timezone.utc),
    'settings': {'notifications': 'true', 'max_items': 100, 'auto_save': False},
}, validated_python
assert [type(validated_python[k]) for k in ('user_id', 'score', 'active')] == [int, float, bool]

# Demonstrate serialization capabilities
# Convert back to JSON
from pydantic_core import to_json
serialized_json = to_json(validated_from_json)
print(f"\n✓ Serialization successful:")
print(f"  JSON length: {len(serialized_json)} bytes")
print(f"  Sample: {serialized_json[:100].decode()}...")

# The datetime serialises as ISO 8601 and the bytes validate back to the same values
assert serialized_json == (b'{"user_id":123,"score":95.5,"active":true,"created_at":"2024-01-15T10:30:00",'
                           b'"settings":{"notifications":"false","max_items":"50","theme":"dark"}}'), serialized_json
assert validator.validate_json(serialized_json) == validated_from_json

print("Example 3 completed successfully!")

In [ ]:
# Example 4: Advanced Schema Features and Custom Validation
print("=== Advanced Schema Features and Custom Validation ===")

# Demonstrate advanced schema features
def validate_email(email: str) -> str:
    """Custom email validation function"""
    if '@' not in email or '.' not in email:
        raise ValueError('Invalid email format')
    return email.lower()

def validate_positive_even(value: int) -> int:
    """Custom validation for positive even numbers"""
    if value <= 0:
        raise ValueError('Value must be positive')
    if value % 2 != 0:
        raise ValueError('Value must be even')
    return value

# Advanced schema with custom validators and complex types
advanced_schema = core_schema.typed_dict_schema({
    'id': core_schema.typed_dict_field(core_schema.no_info_after_validator_function(
        validate_positive_even,
        core_schema.int_schema()
    )),
    'email': core_schema.typed_dict_field(core_schema.no_info_after_validator_function(
        validate_email,
        core_schema.str_schema()
    )),
    'tags': core_schema.typed_dict_field(core_schema.set_schema(core_schema.str_schema())),  # Set instead of list
    'coordinates': core_schema.typed_dict_field(core_schema.tuple_schema([
        core_schema.float_schema(),
        core_schema.float_schema()
    ])),
    'status': core_schema.typed_dict_field(core_schema.literal_schema(['pending', 'active', 'inactive'])),
    'config': core_schema.typed_dict_field(core_schema.union_schema([
        core_schema.str_schema(),
        core_schema.dict_schema(
            keys_schema=core_schema.str_schema(),
            values_schema=core_schema.any_schema()
        )
    ])),
    'optional_field': core_schema.typed_dict_field(
        core_schema.with_default_schema(
            core_schema.nullable_schema(core_schema.str_schema()),
            default=None
        ),
        required=False
    )
})

validator = SchemaValidator(advanced_schema)

# Test advanced validation features
test_data = {
    'id': 42,  # Positive even number
    'email': 'TEST@EXAMPLE.COM',  # Should be lowercased
    'tags': ['python', 'validation', 'rust', 'python'],  # Duplicates should be removed in set
    'coordinates': [40.7128, -74.0060],  # NYC coordinates
    'status': 'active',
    'config': {'debug': True, 'timeout': 30},
    'optional_field': 'some value'
}

print("Testing advanced schema validation:")
validated_advanced = validator.validate_python(test_data)
print("✓ Advanced validation successful:")
print(f"  ID (even/positive): {validated_advanced['id']}")
print(f"  Email (normalized): {validated_advanced['email']}")
print(f"  Tags (set): {validated_advanced['tags']} (type: {type(validated_advanced['tags']).__name__})")
print(f"  Coordinates: {validated_advanced['coordinates']}")
print(f"  Config: {validated_advanced['config']}")
# The custom validators ran (email lower-cased), the list became a de-duplicated set and the pair a tuple
assert validated_advanced == {
    'id': 42, 'email': 'test@example.com', 'tags': {'python', 'validation', 'rust'},
    'coordinates': (40.7128, -74.006), 'status': 'active', 'config': {'debug': True, 'timeout': 30},
    'optional_field': 'some value',
}, validated_advanced

# Test custom validation failures
invalid_advanced_data = [
    {'id': 41, 'email': 'valid@test.com', 'tags': ['test'], 'coordinates': [0, 0], 'status': 'active', 'config': 'simple'},  # Odd ID
    {'id': 42, 'email': 'invalid-email', 'tags': ['test'], 'coordinates': [0, 0], 'status': 'active', 'config': 'simple'},  # Invalid email
    {'id': 42, 'email': 'valid@test.com', 'tags': ['test'], 'coordinates': [0, 0], 'status': 'invalid', 'config': 'simple'}  # Invalid status
]

print(f"\nTesting {len(invalid_advanced_data)} invalid advanced cases:")
expected_first_errors = [('value_error', ('id',)), ('value_error', ('email',)), ('literal_error', ('status',))]
for i, (invalid_data, expected) in enumerate(zip(invalid_advanced_data, expected_first_errors, strict=True), 1):
    try:
        validator.validate_python(invalid_data)
    except ValidationError as e:
        first_error = e.errors()[0]
        print(f"✓ Case {i}: Failed as expected - {first_error['type']} at {first_error['loc']}")
        assert (first_error['type'], first_error['loc']) == expected, (i, first_error)
    else:
        raise AssertionError(f"Case {i}: invalid data passed validation: {invalid_data}")

# Demonstrate schema introspection
print(f"\nSchema introspection:")
print(f"  Schema type: {type(advanced_schema)}")
print(f"  Schema keys: {list(advanced_schema.get('fields', {}).keys()) if isinstance(advanced_schema, dict) else 'N/A'}")

print("Example 4 completed successfully!")

In [ ]:
# Example 5: Performance Testing and Production Usage Patterns
import time
from typing import List, Dict, Any

print("=== Performance Testing and Production Usage ===")

# Create a realistic production schema
user_profile_schema = core_schema.typed_dict_schema({
    'user_id': core_schema.typed_dict_field(core_schema.int_schema(ge=1)),
    'username': core_schema.typed_dict_field(core_schema.str_schema(min_length=3, max_length=50)),
    'email': core_schema.typed_dict_field(core_schema.str_schema()),
    'profile': core_schema.typed_dict_field(core_schema.typed_dict_schema({
        'first_name': core_schema.typed_dict_field(core_schema.str_schema(max_length=100)),
        'last_name': core_schema.typed_dict_field(core_schema.str_schema(max_length=100)),
        'age': core_schema.typed_dict_field(core_schema.int_schema(ge=13, le=120)),
        'interests': core_schema.typed_dict_field(core_schema.list_schema(core_schema.str_schema())),
        'settings': core_schema.typed_dict_field(core_schema.dict_schema(
            keys_schema=core_schema.str_schema(),
            values_schema=core_schema.union_schema([
                core_schema.str_schema(),
                core_schema.int_schema(),
                core_schema.bool_schema()
            ])
        ))
    })),
    'metadata': core_schema.typed_dict_field(core_schema.dict_schema(
        keys_schema=core_schema.str_schema(),
        values_schema=core_schema.any_schema()
    )),
    'created_at': core_schema.typed_dict_field(core_schema.datetime_schema()),
    'is_active': core_schema.typed_dict_field(core_schema.bool_schema())
})

validator = SchemaValidator(user_profile_schema)

# Generate test data for performance testing
def generate_test_user(user_id: int) -> Dict[str, Any]:
    return {
        'user_id': user_id,
        'username': f'user_{user_id}',
        'email': f'user_{user_id}@example.com',
        'profile': {
            'first_name': f'User',
            'last_name': f'Number{user_id}',
            'age': 25 + (user_id % 40),
            'interests': ['coding', 'reading', 'gaming'][:((user_id % 3) + 1)],
            'settings': {
                'theme': 'dark' if user_id % 2 else 'light',
                'notifications': user_id % 2 == 0,
                'max_posts': 50 + (user_id % 100)
            }
        },
        'metadata': {
            'source': 'api',
            'version': '2.0',
            'flags': {'premium': user_id % 5 == 0}
        },
        'created_at': f'2024-01-{(user_id % 28) + 1:02d}T10:00:00',
        'is_active': True
    }

# Performance test: validate many records
test_count = 1000
print(f"Generating {test_count} test records...")
test_data = [generate_test_user(i) for i in range(1, test_count + 1)]

# Validate Python objects
print(f"Validating {test_count} Python objects...")
start_time = time.time()
successful_validations = 0
validation_errors = 0

for i, data in enumerate(test_data):
    try:
        validated = validator.validate_python(data)
        successful_validations += 1
    except ValidationError:
        validation_errors += 1

python_duration = time.time() - start_time
python_rate = test_count / python_duration

print(f"✓ Python validation completed:")
print(f"  Duration: {python_duration:.3f} seconds")
print(f"  Rate: {python_rate:.1f} validations/second")
print(f"  Success: {successful_validations}/{test_count}")
print(f"  Errors: {validation_errors}")
assert (successful_validations, validation_errors) == (test_count, 0), (successful_validations, validation_errors)

# Test JSON validation performance
print(f"\nTesting JSON validation performance...")
import json
json_test_data = [json.dumps(data) for data in test_data[:100]]  # Smaller sample for JSON

start_time = time.time()
json_successful = 0
json_errors = 0

for json_str in json_test_data:
    try:
        validated = validator.validate_json(json_str)
        json_successful += 1
    except ValidationError:
        json_errors += 1

json_duration = time.time() - start_time
json_rate = len(json_test_data) / json_duration

print(f"✓ JSON validation completed ({len(json_test_data)} samples):")
print(f"  Duration: {json_duration:.3f} seconds")
print(f"  Rate: {json_rate:.1f} validations/second")
print(f"  Success: {json_successful}/{len(json_test_data)}")
assert (json_successful, json_errors) == (len(json_test_data), 0), (json_successful, json_errors)

# Demonstrate error aggregation and reporting
print(f"\nError handling patterns:")
invalid_test_data = {
    'user_id': 0,  # Invalid: too small
    'username': 'ab',  # Invalid: too short
    'email': 'invalid',  # Invalid: bad format
    'profile': {
        'first_name': 'Test',
        'last_name': 'User',
        'age': 200,  # Invalid: too old
        'interests': ['valid'],
        'settings': {'theme': 'light'}
    },
    'metadata': {},
    'created_at': 'invalid-date',  # Invalid: bad format
    'is_active': 'not-boolean'  # Invalid: wrong type
}

try:
    validator.validate_python(invalid_test_data)
except ValidationError as e:
    print(f"Validation failed with {len(e.errors())} errors:")
    for error in e.errors()[:3]:  # Show first 3 errors
        print(f"  • {error['type']} at {'.'.join(str(x) for x in error['loc'])}: {error['msg']}")
    # Every invalid field is reported in one pass ('email' is a plain str schema, so 'invalid' is accepted)
    assert [(error['type'], error['loc']) for error in e.errors()] == [
        ('greater_than_equal', ('user_id',)), ('string_too_short', ('username',)), ('less_than_equal', ('profile', 'age')),
        ('datetime_from_date_parsing', ('created_at',)), ('bool_parsing', ('is_active',)),
    ], e.errors()
else:
    raise AssertionError("invalid_test_data passed validation")

# Show pydantic_core capabilities
print(f"\npydantic_core capabilities:")
print("✓ High-performance Rust-based validation")
print("✓ JSON and Python object validation")
print("✓ Automatic type coercion and conversion")
print("✓ Custom validator functions")
print("✓ Complex nested schema support")
print("✓ Comprehensive error reporting")
print("✓ Production-ready performance")

print("Example 5 completed successfully!")